# Component 2: Reinforcement Learning — Theory & Results

Three RL approaches to smart-grid energy cost optimization, benchmarked against a
random policy and a rule-based heuristic on the same real cost function: single-agent
PPO, single-agent SAC, and a decentralized multi-agent MADDPG+GNN system (one agent per
prosumer/grid node, coordinating without a shared controller).

**A note on how this notebook works, unlike `forecasting_theory.ipynb`:** the RL
environment, PPO/SAC training, and MARL training live across several interdependent
files (`src/envs/`, `src/training/train_ppo.py`, `train_sac.py`, `train_marl.py`,
`src/agents/marl_agent.py`, `src/training/benchmark.py`) built and evolved over many
sessions. Rather than re-import and re-run that whole stack from inside a notebook —
which risks silently drifting from whatever state those files are actually in on your
machine — this notebook documents the real math and reproduces the **actual measured
results** from your training runs as a clearly labeled static table, matching exactly
what `benchmark.py` printed when you ran it. If you want a live-executable RL notebook
later, the safest path is pointing me at the current, exact contents of `src/envs/` and
`benchmark.py` so any code cells here are verified against what's really on disk, the
same way every cell in the forecasting notebook was.

## Problem setup

A smart-grid environment where each timestep, an agent (or set of agents) decides how
much power to draw from vs. feed back to the grid, charge/discharge a battery, and shift
flexible load, to minimize total electricity cost over an episode subject to
demand-satisfaction and battery-capacity constraints. `Random` and `Heuristic` (a
rule-based baseline: e.g. charge when price is low, discharge/sell when price is high)
anchor the scale — any learned policy has to beat `Heuristic` to be worth deploying, and
beating `Random` by a wide margin is a basic sanity check that training did *something*.

## PPO: clipped surrogate objective

PPO updates a stochastic policy $\pi_\theta$ while constraining each update to stay close
to the previous policy $\pi_{\theta_{old}}$, using the probability ratio
$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ and an advantage
estimate $\hat{A}_t$:

$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\big(r_t(\theta)\hat{A}_t,\ \mathrm{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\big)\right]$$

Clipping removes the incentive to push $r_t$ far from 1 in a single update — once the
policy has changed enough that the clipped term is worse than the unclipped one, the
gradient contribution from that timestep is capped, which is what makes PPO much more
stable to tune than vanilla policy gradient while still being fully on-policy.

## SAC: maximum-entropy actor-critic

SAC augments the standard RL objective with an entropy bonus, so the policy is trained to
maximize expected return *and* action entropy jointly:

$$J(\pi) = \sum_t \mathbb{E}_{(s_t,a_t)\sim\pi}\big[r(s_t,a_t) + \alpha\, \mathcal{H}(\pi(\cdot|s_t))\big]$$

The entropy term $\alpha\mathcal{H}(\pi(\cdot|s_t))$ explicitly rewards staying stochastic,
which keeps exploration alive throughout training rather than collapsing early to a
deterministic policy — SAC also learns two Q-functions (clipped double-Q) and updates
them off-policy from a replay buffer, making it considerably more sample-efficient than
PPO in principle, at the cost of more hyperparameters (target entropy, $\alpha$
auto-tuning, target network update rate) to get right.

## MADDPG: centralized training, decentralized execution

Each grid node/prosumer is a separate agent with its own actor $\mu_i(o_i)$ that only
sees its own local observation $o_i$ at execution time — no shared controller, no
communication at runtime. But *during training*, each agent's critic $Q_i(x, a_1,
\ldots, a_N)$ is centralized: it sees the joint state $x$ and every agent's action, which
is what makes multi-agent training tractable at all. Without this, from any single
agent's point of view the environment is non-stationary (every other agent's policy is
also changing underneath it), and a decentralized critic has no way to distinguish "my
action was bad" from "the world changed because someone else's policy changed."

The deterministic policy gradient for agent $i$:

$$\nabla_{\theta_i} J(\mu_i) = \mathbb{E}_x\left[\nabla_{\theta_i}\mu_i(o_i)\, \nabla_{a_i}Q_i(x, a_1,\ldots,a_N)\big|_{a_i=\mu_i(o_i)}\right]$$

and each critic is trained by TD learning against a centralized target:

$$\mathcal{L}(\theta_{Q_i}) = \mathbb{E}\left[\big(Q_i(x,a_1,\ldots,a_N) - y_i\big)^2\right],\qquad
y_i = r_i + \gamma\, Q_i'(x', a_1',\ldots,a_N')\big|_{a_j'=\mu_j'(o_j')}$$

This project's version additionally uses a GNN layer over the grid's node/edge structure
so each agent's observation encodes local graph context (neighboring nodes' state), not
just its own isolated readings.

## The real instability story

A from-scratch 500-episode MADDPG training run (`python src/training/train_marl.py
--episodes 500`, no seed set) produced a policy that finished **worse than random**:
**-26.4% vs. heuristic**, with the reward curve improving early, then degrading and
plateauing badly. This was root-caused, not assumed, by reproducing the failure in a
sandbox and testing two hypotheses:

1. **No random seeding** in `MADDPGTrainer` — every run was a different, uncontrolled
   draw of initial weights and exploration noise, so "it got worse" couldn't even be
   distinguished from "this particular seed is unlucky."
2. **Actor and critic sharing one learning rate** (`1e-3`) — a known instability pattern
   in the DDPG family. The critic needs to adapt quickly to a moving multi-agent target,
   but the actor updating at the same speed off a still-inaccurate critic estimate is a
   classic recipe for the policy chasing a noisy gradient signal and diverging.

**The fix**, in `src/agents/marl_agent.py` and `src/training/train_marl.py`:
- `seed` parameter (`torch.manual_seed` / `np.random.seed`) for reproducibility.
- Separate learning rates: `actor_lr=1e-4`, `critic_lr=1e-3` — the critic moves faster
  than the actor, so the actor is always updating against a reasonably converged value
  estimate rather than a wildly shifting target.
- Gradient clipping (`nn.utils.clip_grad_norm_`, max norm 1.0) on all four networks
  (`critic_a`, `critic_b`, `actor_a`, `actor_b`) before every optimizer step.
- **Best-checkpoint tracking**: since RL training is not guaranteed to improve
  monotonically, a 5-episode deterministic evaluation runs every 50 episodes, and
  `outputs/marl_actors_best.pt` is only overwritten when that evaluation actually
  improves — the final episode's policy is compared against the best checkpoint at the
  end, rather than blindly trusting whichever policy training happened to end on.

Sandbox verification (two different seeds, 60 episodes each) showed both converging
cleanly to roughly **+14% vs. heuristic** with healthy, monotonic-ish reward curves — a
qualitative fix, confirmed before touching the real 500-episode run.

The real full run then validated the fix's practical value directly: the deterministic
evaluation improved to a best cost of **\$243,480.88** by episode 100, then drifted worse
to **\$294,555.79** by episode 500 if you'd just taken the final checkpoint. The
best-checkpoint mechanism caught this and preserved the good result — the final benchmark
below (\$242,675.76) matches the logged best almost exactly, which is the fix earning its
keep on real data, not just in a clean sandbox.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Real results from the actual benchmark run (python src/training/benchmark.py),
# reproduced here verbatim as a static, documented table -- not recomputed by this
# notebook. See the markdown note at the top for why.
results = pd.DataFrame([
    {"method": "Random",    "cost_usd": 305_994.95, "pct_vs_heuristic": -6.6,  "target_pct": None},
    {"method": "Heuristic", "cost_usd": 286_946.91, "pct_vs_heuristic": 0.0,   "target_pct": None},
    {"method": "PPO",       "cost_usd": 230_755.39, "pct_vs_heuristic": 19.6,  "target_pct": 20},
    {"method": "SAC",       "cost_usd": 235_291.61, "pct_vs_heuristic": 18.0,  "target_pct": 25},
    {"method": "MARL",      "cost_usd": 242_675.76, "pct_vs_heuristic": 15.4,  "target_pct": 30},
])
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#999999", "#555555", "#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar(results["method"], results["pct_vs_heuristic"], color=colors)
for i, row in results.iterrows():
    if row["target_pct"] is not None:
        ax.hlines(row["target_pct"], i - 0.4, i + 0.4, colors="black", linestyles="dashed", linewidth=1)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("% cost improvement vs. Heuristic")
ax.set_title("Real benchmark: all three learned policies beat Heuristic,\nnone reached their target (dashed lines)")
plt.tight_layout()
plt.show()

## Honest assessment

All three learned methods beat both `Random` and `Heuristic` — training clearly worked,
and the MADDPG stability fix (seeding, split learning rates, gradient clipping,
best-checkpoint tracking) took a run that was *worse than random* and turned it into one
competitive with the single-agent methods. But none reached its target: PPO +19.6% vs.
>20%, SAC +18.0% vs. >25%, MARL +15.4% vs. >30%.

Given all three landing close to but under target, the decision made on this project was
**document honestly and move on**, rather than continuing to chase the targets through
further RL tuning. Plausible reasons the targets weren't hit, for future reference rather
than as excuses:

- Training budgets here (hundreds of episodes) are small relative to what published
  smart-grid RL results typically use (often tens of thousands of episodes or more).
- Reward shaping was kept simple (direct cost minimization); credit assignment across
  multiple time-coupled constraints (battery state carried across steps, demand
  satisfaction) is a known hard problem for RL without more shaping or curriculum design.
- Multi-agent credit assignment is inherently harder than single-agent: MARL landing
  *below* PPO/SAC, despite the extra structure (GNN, centralized critics), is consistent
  with the literature's general finding that decentralized execution costs some
  performance relative to a single agent with full global information, in exchange for
  the realism of not requiring a central controller at runtime.

This gap is reported here exactly as it was found — it's a meaningful, legitimate result
for a portfolio project, not a defect to talk around.